# 02 — ML Risk Model (Heart Disease)

Gradient-boosting classifier for binary heart-disease prediction.

**Lineage.**
- Scikit-learn `Pipeline` + scaling discipline → Project 3 (UCI Online Retail II K-Means RFM).
- Per-slice disaggregated evaluation by `sex` and age band → Project 4 (Fashion-MNIST CNN with dropout), where same headline accuracy hid large per-class differences.

**Educational artifact only. Not for clinical use.**

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    roc_curve, precision_recall_curve, confusion_matrix, classification_report
)
from sklearn.inspection import permutation_importance

from src.data_loader import load_heart_disease
from src.preprocessing import split_and_preprocess
from src.ml_model import train_and_evaluate, predict_proba, save, slice_metrics

## 1. Train + cross-validate

In [ ]:
df = load_heart_disease()
split = split_and_preprocess(df)
model, metrics = train_and_evaluate(split)
print(metrics)
save(model)

## 2. ROC and Precision-Recall curves

In [ ]:
probs = predict_proba(model, split.X_test)
fpr, tpr, _ = roc_curve(split.y_test, probs)
prec, rec, _ = precision_recall_curve(split.y_test, probs)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(fpr, tpr); axes[0].plot([0, 1], [0, 1], '--', color='gray')
axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')
axes[0].set_title(f'ROC (AUC = {metrics.test_roc_auc:.3f})')
axes[1].plot(rec, prec)
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].set_title(f'PR (AUC = {metrics.test_pr_auc:.3f})')
plt.tight_layout(); plt.show()

## 3. Confusion matrix at threshold 0.5

In [ ]:
y_pred = (probs >= 0.5).astype(int)
cm = confusion_matrix(split.y_test, y_pred)
print(cm)
print(classification_report(split.y_test, y_pred, digits=3))

## 4. Permutation feature importance
Permutation importance is computed on the held-out test set so we measure what the model uses to generalise, not what it used to fit.

In [ ]:
result = permutation_importance(
    model, split.X_test, split.y_test, n_repeats=20, random_state=42, scoring='roc_auc'
)
imp = pd.DataFrame({
    'feature': split.X_test.columns,
    'importance': result.importances_mean,
    'std': result.importances_std,
}).sort_values('importance', ascending=True)
fig, ax = plt.subplots(figsize=(6, 5))
ax.barh(imp['feature'], imp['importance'], xerr=imp['std'], color='#4C9AFF')
ax.set_xlabel('Drop in ROC-AUC when shuffled')
ax.set_title('Permutation importance (test set)')
plt.tight_layout(); plt.show()

## 5. Per-slice metrics — bias audit
Lineage: P4 surfaced that aggregate accuracy can hide large per-class differences (Coat +9.4, Shirt -6.1 between identical-headline models). The same disaggregation discipline applied here checks whether the headline ROC-AUC hides cohort-level disparities.

In [ ]:
print('--- By sex (0 = female, 1 = male) ---')
print(slice_metrics(model, split.X_test, split.y_test, 'sex'))

In [ ]:
# Age-band slice
X_test_aug = split.X_test.copy()
X_test_aug['age_band'] = pd.cut(
    X_test_aug['age'], bins=[0, 45, 55, 65, 120],
    labels=['<45', '45-54', '55-64', '65+']
)
print('--- By age band ---')
print(slice_metrics(model, X_test_aug, split.y_test, 'age_band'))

## 6. Notes for the synthesis paper
- Headline ROC-AUC is strong on this small public cohort, but slice sizes are tiny — slice AUCs are point estimates with wide implicit confidence bands.
- The disaggregation discipline is the point, not the headline number; it carries forward into Phase J (Evaluation).
- The trained model is persisted under `models/ml_model.joblib` (gitignored) and consumed by the agent orchestrator in Phase H.